# MACHINE LEARNING 2026 - FINAL PROJECT (Option 1)

Course 2025/26: *C. Sun*, *M. Pavan*, *P. Zanuttigh*  

**Group Members:**        
*`Tommaso Dambi ID:2141118`*         
*`Alessandro Costabile ID:2146702`*        
*`Carlo Toffoli ID:2144522`*

## Heart Disease Analysis Using Clustering and Classification
<center>
    <img src="data/dataset-cover.jpg" style = "width: 50%;">
</center>

The dataset labels are the following:

|   id| age   | sex | dataset | cp | trestbps | chol | fbs | restecg | thalch | exang | oldpeak |slope | ca | thal | num |
| :-: | :-:  |  :-: | :-: | :-:| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| 1  |  63   | Male | Cleveland | typical angina | 145 | 233 | TRUE | Iv hypertrophy | 150 | FALSE | 2.3 | downsloping | 0 | fixed defect | 0 |
| 2  | 67   | Male | Cleveland | asymptomatic | 160 | 286 | FALSE | Iv hypertrophy | 108 | TRUE | 1.5 | flat | 3 | normal | 1 |
| 3   | 67 | Male | Cleveland | asymptomatic | 120 | 229 | FALSE | Iv hypertrophy | 129 | TRUE | 2.6 | flat | 2 | reversable | 2 |
| 4   | 37| Male | Cleveland | non-anginal | 130 | 250 | FALSE | normal | 187 | FALSE | 3.5 | downsloping | 0 | normal | 3 |

Description of the features:
 - `cp`: chest pain type:
    1. typical angina
    2. atypical angina
    3. non-anginal pain
    4. asymptomatic
 - `trestbps`: resting blood pressure (in mm Hg on admission to the hospital)
 - `chol`: serum cholestoral [mg/dL]
 - `fbs`: fasting blood sugar (True if above 120 mg/dL)
 - `restecg`: resting electrocardiographic results
    1. having ST-T wave abnormality (T wave inversions and/or ST elevation or depression of > 0.05 mV)
    2. showing probable or definite left ventricular hypertrophy by Estes' criteria
    3. normal
 - `thalch`: maximum heart rate achieved
 - `exang`: exercise induced angina
    1. yes
    2. no
 - `oldpeak`: ST depression induced by exercise relative to rest
 - `slope`: the slope of the peak exercise ST segment
    1. upsloping
    2. flat
    3. downsloping
 - `ca`: number of major vessels (0-3) colored by flouroscopy
 - `thal`: type of defect in blood flow towards heart
    1. fixed defect
    2. reversable
    3. normal
 - `num`: label - severity of the disease from 0 (not present) to 4 (max severity)

Some records lack of data of some features such as `trestbps`, `chol`, `fbs`, `thalch`, `exang`, `oldpeak`, `ca`. This has to be addressed, maybe with clustering?

# ⚠️ Attenzione
Il dataset non è incluso per ovvi motivi nel repository github. Scaricalo, estrailo e mettilo nella cartella `/data`. Ricorda di aggiornare il nome del file!
Also: the jupyter server was launched from the upper dir of the project (tdc_fp/..). All the relative imports were done accordingly.

# CODE

## 1. Data Preprocessing
All the preprocessing helper functions are defined in `preproc.py`.
As many samples lack the significant `ca` feature we have decided to use just the Cleveland data which provides that feature in almost all the samples.

In [ ]:
# print(feature_names)
import pandas as pd
from seaborn import pairplot
df = pd.DataFrame(X)
df['num'] = Y

pairplot(df, hue="num", corner=True)
# plt.savefig("pairplot_onehot.jpg")

# heatmap(data=dataframe.corr(), cmap="crest")
# plt.savefig("heatmap.jpg")

## 2. Clustering analysis

Va capita l'utilità di questa cosa: una volta trovati i cluster cosa ce ne facciamo? Vogliamo classificare in maniera unsupervised? boh
Ok, poi devo farlo prima del PCA? PCA serve solo a visualizzare i dati o posso anche comprendere se una componente in particolare ha la giusta combinazione di features che permette una classificazione lineare?

In [345]:
# Dataset split
permuted_indexes = np.random.permutation(X.shape[0])

X = X[permuted_indexes]
Y = Y[permuted_indexes].ravel()

# Splitting with stratification ensures consistency of the Y distribution between the train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=ID)
print("Distribution of patients in train set:", np.unique(y_train, return_counts=True)[1])
print("Distribution of patients in test set:", np.unique(y_test, return_counts=True)[1])

Distribution of patients in train set: [128 112]
Distribution of patients in test set: [33 28]


In [1]:
# For what concerns visualization, we need to decompose the data into at most 3 components
# (plus the labels which are represented by the color of the datapoints)

# Do clustering into two clusters
clustering = KMeans(2, n_init=10).fit(X)
# clustering = AgglomerativeClustering(2, linkage='ward').fit(X)

labels = clustering.labels_

print("Scores - also with inverted labels")
print(f"Accuracy: {round(accuracy_score(labels, Y)*100, 2)} | {round((1-accuracy_score(labels, Y))*100, 2)} %")
print(f"Precision: {round(precision_score(labels, Y)*100, 2)} | {round((1-precision_score(labels, Y))*100, 2)} %")
# print(f"Inertia: {clustering.inertia_}")

pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)

plot_clusters(X_reduced, (Y == labels).astype(int), "Correctly classified")
plot_clusters(X_reduced, labels, "Cluster labels")
plot_clusters(X_reduced, Y, "Dataset labels")

print("PCA weights:")
print(pca.components_[0])

NameError: name 'KMeans' is not defined

## Clustering Analysis
Martina propone un clustering multiclasse.
Al variare del numero di cluster notare:
- I cluster a maggior rischio (quelli con il numero piu' elevato di malati pesati con il rischio, e normalizzati alla somma dei pesi)
- (min, max, mean) dei valori assunti dalle features in quei cluster

In [1]:
from clustering import algo_showcase
from sklearn.cluster import KMeans, AgglomerativeClustering
from preproc import Preprocess, filter_dataset
import pandas as pd

pd.set_option('display.max_columns', None)

ID = 2144522

data = Preprocess(
    dropped=['dataset'],
    ordinal=True,
    encoded_features=[1,2,6,10,12],
    permute_map=True,
    permute_data=True,
    multiclass=True,
    random_state=ID
)

algo = KMeans(n_init=1, random_state = ID)
# algo = AgglomerativeClustering(linkage='ward')

for df in algo_showcase(data, range(6,15), algo, head=1):
    df.set_index('cluster n°', inplace=True)
    display(df)

Dataset is float64 (299, 13)


Number of clusters: 6
Risk variance: 0.56
Risk maxima: 0.30


,risk fom,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal
cluster n°,,,,,,,,,,,,,,
1,0.301,"(35.0, 70.0, 56.259)","(0.0, 1.0, 0.103)","(0.0, 3.0, 0.155)","(100.0, 170.0, 128.845)","(100.0, 409.0, 246.776)","(0.0, 0.0, 0.0)","(0.0, 2.0, 1.466)","(71.0, 169.0, 125.259)","(0.0, 1.0, 0.966)","(0.0, 5.6, 1.924)","(0.0, 2.0, 1.879)","(0.0, 3.0, 1.017)","(0.0, 2.0, 1.103)"
4,0.160,"(41.0, 74.0, 58.136)","(0.0, 1.0, 0.939)","(0.0, 3.0, 1.0)","(100.0, 180.0, 133.227)","(149.0, 564.0, 271.197)","(0.0, 1.0, 0.106)","(1.0, 2.0, 1.455)","(96.0, 179.0, 152.121)","(0.0, 1.0, 0.167)","(0.0, 2.6, 0.503)","(0.0, 2.0, 0.758)","(0.0, 2.0, 0.333)","(1.0, 2.0, 1.97)"
3,0.151,"(29.0, 59.0, 45.544)","(0.0, 1.0, 0.177)","(0.0, 3.0, 1.392)","(94.0, 192.0, 125.038)","(141.0, 325.0, 230.228)","(0.0, 1.0, 0.038)","(1.0, 2.0, 1.722)","(123.0, 202.0, 168.405)","(0.0, 1.0, 0.051)","(0.0, 3.8, 0.428)","(0.0, 2.0, 0.392)","(0.0, 2.0, 0.165)","(0.0, 2.0, 1.759)"
2,0.113,"(46.0, 76.0, 59.75)","(0.0, 1.0, 0.225)","(0.0, 3.0, 0.825)","(112.0, 180.0, 140.125)","(164.0, 407.0, 248.45)","(0.0, 0.0, 0.0)","(0.0, 2.0, 1.175)","(97.0, 173.0, 143.2)","(0.0, 1.0, 0.025)","(0.2, 6.2, 2.055)","(0.0, 2.0, 1.75)","(0.0, 3.0, 1.15)","(0.0, 2.0, 1.125)"
5,0.088,"(43.0, 69.0, 57.879)","(0.0, 1.0, 0.152)","(0.0, 3.0, 0.788)","(108.0, 200.0, 143.212)","(126.0, 341.0, 242.697)","(1.0, 1.0, 1.0)","(1.0, 2.0, 1.394)","(90.0, 178.0, 147.091)","(0.0, 1.0, 0.394)","(0.0, 4.0, 1.327)","(0.0, 2.0, 1.182)","(0.0, 3.0, 1.182)","(0.0, 2.0, 1.212)"
0,0.076,"(35.0, 77.0, 56.696)","(0.0, 0.0, 0.0)","(0.0, 3.0, 0.261)","(94.0, 150.0, 126.435)","(149.0, 304.0, 236.609)","(0.0, 0.0, 0.0)","(1.0, 2.0, 1.478)","(111.0, 186.0, 150.348)","(0.0, 1.0, 0.609)","(0.0, 1.9, 0.517)","(0.0, 0.0, 0.0)","(0.0, 3.0, 0.957)","(0.0, 2.0, 1.174)"




Number of clusters: 7
Risk variance: 0.80
Risk maxima: 0.30


,risk fom,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal
cluster n°,,,,,,,,,,,,,,
1,0.298,"(35.0, 70.0, 55.945)","(0.0, 1.0, 0.055)","(0.0, 2.0, 0.109)","(100.0, 170.0, 129.491)","(100.0, 353.0, 242.455)","(0.0, 0.0, 0.0)","(0.0, 2.0, 1.491)","(71.0, 154.0, 122.927)","(0.0, 1.0, 0.945)","(0.0, 5.6, 1.947)","(0.0, 2.0, 1.873)","(0.0, 3.0, 1.018)","(0.0, 2.0, 1.073)"
4,0.220,"(39.0, 76.0, 57.045)","(0.0, 1.0, 0.94)","(0.0, 3.0, 0.97)","(100.0, 180.0, 132.239)","(149.0, 360.0, 255.881)","(0.0, 1.0, 0.03)","(0.0, 2.0, 1.537)","(96.0, 179.0, 151.269)","(0.0, 1.0, 0.164)","(0.0, 2.6, 0.507)","(0.0, 2.0, 0.896)","(0.0, 2.0, 0.299)","(1.0, 2.0, 1.97)"
3,0.217,"(29.0, 59.0, 45.682)","(0.0, 1.0, 0.136)","(0.0, 3.0, 1.333)","(94.0, 154.0, 124.515)","(141.0, 325.0, 231.348)","(0.0, 1.0, 0.045)","(1.0, 2.0, 1.727)","(123.0, 202.0, 170.045)","(0.0, 1.0, 0.045)","(0.0, 3.5, 0.361)","(0.0, 2.0, 0.152)","(0.0, 2.0, 0.167)","(1.0, 2.0, 1.833)"
5,0.092,"(43.0, 71.0, 58.316)","(0.0, 1.0, 0.263)","(0.0, 3.0, 0.921)","(108.0, 200.0, 141.947)","(126.0, 417.0, 252.132)","(1.0, 1.0, 1.0)","(1.0, 2.0, 1.342)","(90.0, 178.0, 147.737)","(0.0, 1.0, 0.368)","(0.0, 4.0, 1.2)","(0.0, 2.0, 1.026)","(0.0, 3.0, 1.158)","(0.0, 2.0, 1.316)"
2,0.088,"(38.0, 69.0, 55.757)","(0.0, 0.0, 0.0)","(0.0, 3.0, 1.324)","(110.0, 192.0, 136.297)","(185.0, 293.0, 236.541)","(0.0, 0.0, 0.0)","(1.0, 2.0, 1.297)","(103.0, 195.0, 152.081)","(0.0, 1.0, 0.054)","(0.0, 4.2, 1.589)","(0.0, 2.0, 1.73)","(0.0, 3.0, 0.676)","(0.0, 2.0, 1.054)"
0,0.076,"(35.0, 77.0, 56.696)","(0.0, 0.0, 0.0)","(0.0, 3.0, 0.261)","(94.0, 150.0, 126.435)","(149.0, 304.0, 236.609)","(0.0, 0.0, 0.0)","(1.0, 2.0, 1.478)","(111.0, 186.0, 150.348)","(0.0, 1.0, 0.609)","(0.0, 1.9, 0.517)","(0.0, 0.0, 0.0)","(0.0, 3.0, 0.957)","(0.0, 2.0, 1.174)"
6,0.041,"(55.0, 70.0, 61.923)","(0.0, 1.0, 0.846)","(0.0, 1.0, 0.077)","(114.0, 180.0, 141.385)","(164.0, 564.0, 328.154)","(0.0, 0.0, 0.0)","(0.0, 2.0, 0.923)","(109.0, 160.0, 140.846)","(0.0, 1.0, 0.231)","(1.0, 6.2, 2.715)","(1.0, 2.0, 1.769)","(0.0, 3.0, 1.769)","(0.0, 2.0, 1.308)"




Number of clusters: 8
Risk variance: 0.46
Risk maxima: 0.24


...

,risk fom,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal
cluster n°,,,,,,,,,,,,,,
7,0.174,"(43.0, 68.0, 58.737)","(0.0, 1.0, 0.263)","(0.0, 1.0, 0.211)","(117.0, 200.0, 146.0)","(176.0, 341.0, 254.211)","(1.0, 1.0, 1.0)","(1.0, 2.0, 1.368)","(90.0, 165.0, 137.421)","(0.0, 1.0, 0.684)","(0.0, 4.0, 1.726)","(0.0, 2.0, 1.526)","(0.0, 3.0, 1.474)","(0.0, 2.0, 0.947)"
1,0.159,"(35.0, 70.0, 55.412)","(0.0, 1.0, 0.029)","(0.0, 1.0, 0.059)","(100.0, 150.0, 122.147)","(100.0, 299.0, 214.647)","(0.0, 0.0, 0.0)","(0.0, 2.0, 1.324)","(71.0, 154.0, 124.559)","(0.0, 1.0, 0.912)","(0.0, 5.6, 2.171)","(0.0, 2.0, 1.824)","(0.0, 3.0, 0.824)","(0.0, 2.0, 1.176)"
10,0.152,"(42.0, 70.0, 56.714)","(0.0, 1.0, 0.048)","(0.0, 2.0, 0.19)","(110.0, 170.0, 142.048)","(218.0, 409.0, 292.524)","(0.0, 0.0, 0.0)","(1.0, 2.0, 1.667)","(88.0, 150.0, 120.905)","(0.0, 1.0, 0.952)","(0.0, 4.2, 1.733)","(1.0, 2.0, 1.952)","(0.0, 3.0, 1.429)","(0.0, 2.0, 0.905)"
11,0.108,"(48.0, 74.0, 60.069)","(0.0, 1.0, 0.966)","(0.0, 3.0, 0.828)","(102.0, 180.0, 133.207)","(209.0, 564.0, 303.586)","(0.0, 1.0, 0.034)","(1.0, 2.0, 1.517)","(121.0, 172.0, 156.207)","(0.0, 1.0, 0.103)","(0.0, 1.8, 0.397)","(0.0, 2.0, 0.069)","(0.0, 2.0, 0.414)","(1.0, 2.0, 1.931)"
3,0.089,"(49.0, 70.0, 61.882)","(0.0, 1.0, 0.353)","(0.0, 1.0, 0.294)","(112.0, 160.0, 135.765)","(164.0, 407.0, 259.118)","(0.0, 0.0, 0.0)","(0.0, 2.0, 1.118)","(109.0, 173.0, 143.882)","(0.0, 0.0, 0.0)","(1.0, 6.2, 2.653)","(0.0, 2.0, 1.706)","(1.0, 3.0, 2.294)","(0.0, 2.0, 1.059)"
8,0.087,"(34.0, 63.0, 45.84)","(1.0, 1.0, 1.0)","(0.0, 3.0, 1.48)","(94.0, 160.0, 123.84)","(141.0, 306.0, 216.88)","(0.0, 0.0, 0.0)","(1.0, 2.0, 1.8)","(138.0, 192.0, 167.72)","(0.0, 1.0, 0.04)","(0.0, 1.6, 0.42)","(0.0, 2.0, 0.96)","(0.0, 2.0, 0.24)","(2.0, 2.0, 2.0)"
2,0.082,"(35.0, 59.0, 46.744)","(0.0, 0.0, 0.0)","(0.0, 3.0, 1.372)","(108.0, 152.0, 125.698)","(157.0, 325.0, 234.14)","(0.0, 0.0, 0.0)","(2.0, 2.0, 2.0)","(123.0, 187.0, 163.581)","(0.0, 1.0, 0.093)","(0.0, 3.8, 0.658)","(0.0, 2.0, 0.395)","(0.0, 2.0, 0.14)","(0.0, 2.0, 1.698)"
6,0.080,"(42.0, 76.0, 58.375)","(0.0, 1.0, 0.958)","(0.0, 2.0, 0.375)","(100.0, 180.0, 133.917)","(149.0, 394.0, 254.458)","(0.0, 0.0, 0.0)","(0.0, 2.0, 1.292)","(97.0, 169.0, 139.125)","(0.0, 1.0, 0.458)","(0.0, 3.4, 0.954)","(1.0, 2.0, 1.958)","(0.0, 2.0, 0.167)","(1.0, 2.0, 1.833)"
5,0.070,"(51.0, 70.0, 60.381)","(0.0, 1.0, 0.048)","(1.0, 3.0, 2.524)","(110.0, 192.0, 144.667)","(185.0, 298.0, 242.238)","(0.0, 1.0, 0.19)","(1.0, 2.0, 1.143)","(103.0, 195.0, 150.143)","(0.0, 1.0, 0.095)","(0.0, 4.2, 1.319)","(0.0, 2.0, 1.476)","(0.0, 2.0, 0.381)","(0.0, 2.0, 1.381)"




Number of clusters: 13
Risk variance: 0.14
Risk maxima: 0.16


,risk fom,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal
cluster n°,,,,,,,,,,,,,,
1,0.159,"(35.0, 70.0, 55.412)","(0.0, 1.0, 0.029)","(0.0, 1.0, 0.059)","(100.0, 150.0, 122.147)","(100.0, 299.0, 214.647)","(0.0, 0.0, 0.0)","(0.0, 2.0, 1.324)","(71.0, 154.0, 124.559)","(0.0, 1.0, 0.912)","(0.0, 5.6, 2.171)","(0.0, 2.0, 1.824)","(0.0, 3.0, 0.824)","(0.0, 2.0, 1.176)"
10,0.152,"(42.0, 70.0, 56.714)","(0.0, 1.0, 0.048)","(0.0, 2.0, 0.19)","(110.0, 170.0, 142.048)","(218.0, 409.0, 292.524)","(0.0, 0.0, 0.0)","(1.0, 2.0, 1.667)","(88.0, 150.0, 120.905)","(0.0, 1.0, 0.952)","(0.0, 4.2, 1.733)","(1.0, 2.0, 1.952)","(0.0, 3.0, 1.429)","(0.0, 2.0, 0.905)"
12,0.134,"(55.0, 62.0, 57.75)","(1.0, 1.0, 1.0)","(0.0, 0.0, 0.0)","(160.0, 200.0, 177.5)","(164.0, 327.0, 251.0)","(0.0, 1.0, 0.5)","(0.0, 1.0, 0.75)","(117.0, 146.0, 135.25)","(0.0, 1.0, 0.75)","(2.8, 6.2, 4.1)","(1.0, 2.0, 1.5)","(0.0, 3.0, 1.75)","(0.0, 2.0, 1.0)"
7,0.116,"(43.0, 68.0, 58.556)","(0.0, 1.0, 0.167)","(0.0, 1.0, 0.222)","(108.0, 180.0, 139.556)","(176.0, 341.0, 252.778)","(1.0, 1.0, 1.0)","(1.0, 2.0, 1.444)","(90.0, 165.0, 137.722)","(0.0, 1.0, 0.611)","(0.0, 3.4, 1.45)","(0.0, 2.0, 1.444)","(0.0, 3.0, 1.5)","(0.0, 2.0, 1.0)"
11,0.111,"(48.0, 74.0, 60.133)","(0.0, 1.0, 0.967)","(0.0, 3.0, 0.8)","(102.0, 180.0, 133.433)","(209.0, 564.0, 306.6)","(0.0, 1.0, 0.033)","(1.0, 2.0, 1.5)","(121.0, 172.0, 156.233)","(0.0, 1.0, 0.1)","(0.0, 1.8, 0.423)","(0.0, 2.0, 0.133)","(0.0, 2.0, 0.4)","(1.0, 2.0, 1.933)"
2,0.082,"(35.0, 59.0, 46.744)","(0.0, 0.0, 0.0)","(0.0, 3.0, 1.372)","(108.0, 152.0, 125.698)","(157.0, 325.0, 234.14)","(0.0, 0.0, 0.0)","(2.0, 2.0, 2.0)","(123.0, 187.0, 163.581)","(0.0, 1.0, 0.093)","(0.0, 3.8, 0.658)","(0.0, 2.0, 0.395)","(0.0, 2.0, 0.14)","(0.0, 2.0, 1.698)"
6,0.080,"(42.0, 76.0, 58.087)","(0.0, 1.0, 0.957)","(0.0, 2.0, 0.435)","(100.0, 174.0, 131.739)","(149.0, 307.0, 242.696)","(0.0, 0.0, 0.0)","(0.0, 2.0, 1.348)","(97.0, 169.0, 140.609)","(0.0, 1.0, 0.435)","(0.0, 1.8, 0.8)","(1.0, 2.0, 1.957)","(0.0, 2.0, 0.174)","(1.0, 2.0, 1.826)"
8,0.080,"(34.0, 63.0, 45.783)","(1.0, 1.0, 1.0)","(0.0, 3.0, 1.522)","(94.0, 160.0, 123.826)","(141.0, 306.0, 215.565)","(0.0, 0.0, 0.0)","(1.0, 2.0, 1.87)","(138.0, 192.0, 167.478)","(0.0, 0.0, 0.0)","(0.0, 1.6, 0.452)","(0.0, 2.0, 0.957)","(0.0, 2.0, 0.261)","(2.0, 2.0, 2.0)"
3,0.079,"(49.0, 70.0, 61.875)","(0.0, 1.0, 0.312)","(0.0, 1.0, 0.312)","(112.0, 150.0, 134.25)","(188.0, 407.0, 265.062)","(0.0, 0.0, 0.0)","(0.0, 2.0, 1.125)","(109.0, 173.0, 143.812)","(0.0, 0.0, 0.0)","(1.0, 4.4, 2.431)","(0.0, 2.0, 1.75)","(1.0, 3.0, 2.25)","(0.0, 2.0, 1.062)"




Number of clusters: 14
Risk variance: 0.16
Risk maxima: 0.15


,risk fom,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal
cluster n°,,,,,,,,,,,,,,
10,0.149,"(42.0, 70.0, 56.65)","(0.0, 1.0, 0.05)","(0.0, 2.0, 0.2)","(110.0, 170.0, 141.85)","(226.0, 409.0, 296.25)","(0.0, 0.0, 0.0)","(1.0, 2.0, 1.65)","(88.0, 150.0, 121.7)","(1.0, 1.0, 1.0)","(0.0, 4.2, 1.72)","(1.0, 2.0, 1.95)","(0.0, 3.0, 1.45)","(0.0, 2.0, 0.9)"
12,0.134,"(55.0, 62.0, 57.75)","(1.0, 1.0, 1.0)","(0.0, 0.0, 0.0)","(160.0, 200.0, 177.5)","(164.0, 327.0, 251.0)","(0.0, 1.0, 0.5)","(0.0, 1.0, 0.75)","(117.0, 146.0, 135.25)","(0.0, 1.0, 0.75)","(2.8, 6.2, 4.1)","(1.0, 2.0, 1.5)","(0.0, 3.0, 1.75)","(0.0, 2.0, 1.0)"
1,0.124,"(35.0, 67.0, 54.903)","(0.0, 1.0, 0.032)","(0.0, 3.0, 0.161)","(100.0, 144.0, 119.419)","(100.0, 318.0, 219.29)","(0.0, 0.0, 0.0)","(0.0, 2.0, 1.226)","(96.0, 154.0, 127.032)","(0.0, 1.0, 0.968)","(0.0, 5.6, 2.206)","(1.0, 2.0, 1.871)","(0.0, 3.0, 0.903)","(0.0, 2.0, 1.161)"
11,0.108,"(48.0, 74.0, 59.828)","(0.0, 1.0, 0.966)","(0.0, 2.0, 0.724)","(102.0, 180.0, 133.207)","(209.0, 564.0, 308.931)","(0.0, 1.0, 0.034)","(1.0, 2.0, 1.483)","(121.0, 172.0, 156.414)","(0.0, 1.0, 0.103)","(0.0, 1.6, 0.376)","(0.0, 2.0, 0.138)","(0.0, 2.0, 0.345)","(1.0, 2.0, 1.931)"
7,0.104,"(43.0, 68.0, 58.353)","(0.0, 1.0, 0.118)","(0.0, 1.0, 0.235)","(108.0, 180.0, 139.647)","(176.0, 341.0, 250.353)","(1.0, 1.0, 1.0)","(1.0, 2.0, 1.412)","(90.0, 165.0, 139.588)","(0.0, 1.0, 0.647)","(0.0, 3.4, 1.424)","(0.0, 2.0, 1.412)","(0.0, 3.0, 1.412)","(0.0, 2.0, 0.941)"
5,0.086,"(52.0, 70.0, 59.312)","(0.0, 0.0, 0.0)","(1.0, 3.0, 2.438)","(118.0, 192.0, 151.688)","(186.0, 298.0, 244.438)","(0.0, 1.0, 0.188)","(1.0, 2.0, 1.062)","(125.0, 195.0, 158.438)","(0.0, 0.0, 0.0)","(0.0, 4.2, 0.919)","(0.0, 2.0, 1.125)","(0.0, 1.0, 0.188)","(0.0, 2.0, 1.188)"
6,0.085,"(42.0, 66.0, 55.263)","(1.0, 1.0, 1.0)","(0.0, 2.0, 0.474)","(100.0, 174.0, 133.579)","(177.0, 307.0, 247.158)","(0.0, 0.0, 0.0)","(1.0, 2.0, 1.368)","(122.0, 174.0, 147.579)","(0.0, 1.0, 0.526)","(0.0, 1.8, 0.663)","(1.0, 2.0, 1.947)","(0.0, 2.0, 0.211)","(1.0, 2.0, 1.842)"
13,0.085,"(39.0, 68.0, 56.172)","(0.0, 0.0, 0.0)","(0.0, 2.0, 0.448)","(100.0, 150.0, 128.931)","(149.0, 302.0, 229.0)","(0.0, 0.0, 0.0)","(1.0, 2.0, 1.517)","(105.0, 173.0, 148.586)","(0.0, 0.0, 0.0)","(0.0, 3.6, 1.207)","(0.0, 2.0, 1.379)","(0.0, 3.0, 1.034)","(0.0, 2.0, 1.0)"
4,0.083,"(29.0, 59.0, 43.56)","(0.0, 1.0, 0.04)","(0.0, 3.0, 0.8)","(110.0, 152.0, 128.76)","(172.0, 321.0, 246.92)","(0.0, 0.0, 0.0)","(1.0, 2.0, 1.16)","(144.0, 202.0, 173.6)","(0.0, 1.0, 0.12)","(0.0, 2.0, 0.28)","(0.0, 2.0, 0.28)","(0.0, 2.0, 0.2)","(1.0, 2.0, 1.84)"


### Analysis of clusters
To choose an appropriate number of clusters we exploited the `clustering.algo_showcase` function.
This computes the risk (figure of merit) of the different clusters, and by trying multiple amount of clusters we can identify different variances in risk.
We can assume that setups with noticeable difference between cluster risks are preferred, but also the ones with higher risk over the riskiest clusters are important. The values printed before each table were the ones which enabled us to select the most suitable number of clusters (=7 with KMeans).

## 3. Supervised Learning

In [339]:
# # TODO: Riordinare

# # Call the algorithm selection function
# y_pred, y_score = algo_selection(X_train, X_test, y_train, y_test, ID)
# # The precision is the ratio tp / (tp + fp) where tp is the number of true positives and fp the number of false positives.
# # - Precision: di quelli che ho pedetto malati, quanti lo sono davvero
# # - Recall: dei malati veri, quanti ne ho beccati
# # - F1: media di precision e recall
# print("Test set results: ",
#     "\nAccurancy: ", round(100*np.sum(y_test == y_pred) / len(y_test), 3),
#     "\nPrecision: ", round(100*np.dot(y_test, y_pred) / np.sum(y_pred), 3),
#     "\nRecall: ", round(recall_score(y_test, y_pred), 3))

In [3]:
from preproc import Preprocess, filter_dataset
import numpy as np

data = Preprocess(permute_data=False)

print(filter_dataset(data._dataset, lambda x: x[1][2] == data.value_map[2].index('Cleveland')).shape)

Dataset is float64 (299, 14)
(297, 15)
